# Validação pós-correção de `_normalize_timestamps`

**Objetivo:** comparar RoTHP vs HoTHP (corrigido) no protocolo *train short, test long*
com dados sintéticos de Hawkes, nos dois regimes:
- Decaimento **lento** (β ≈ 0.025)
- Decaimento **rápido** (β ≈ 0.50)

**Correção aplicada:** `_normalize_timestamps` agora é um no-op (apenas subtrai o primeiro evento),
preservando os timestamps já normalizados por `to_tensors` (gap médio ≈ 1.0).

**Fatores de extrapolação:** α ∈ {1, 2, 5, 10}


In [ ]:
# ── Setup (Colab) ─────────────────────────────────────────────────────────────
import os, sys, math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('ufc-easytpp'):
        os.system('git clone https://github.com/hugoramos/ufc-easytpp.git')

    # Aplica correção de _normalize_timestamps (Opção A: no-op que preserva timestamps)
    hothp_path = 'ufc-easytpp/easy_tpp/model/torch_model/torch_hothp.py'
    with open(hothp_path, 'r') as f:
        code = f.read()

    OLD = '''    def _normalize_timestamps(self, time_seqs):
        """Map raw timestamps so that the mean inter-event gap is ~1.0.

        Uses prefix-only normalization: event i is divided by the mean gap
        of events 0..i-1, so no future information leaks into position i.

        For position 0: t_shifted[0] == 0 always, divisor = 1 (irrelevant).
        For position i >= 1: divisor = mean(diffs[0..i-1])
                           = cumsum(diffs)[i-1] / i
        """
        B, T = time_seqs.shape
        # Use first event as reference — causally valid and equal to min for sorted seqs.
        t_shifted = time_seqs - time_seqs[:, :1]

        if T <= 1:
            return t_shifted

        diffs = t_shifted[:, 1:] - t_shifted[:, :-1]          # [B, T-1]
        cumsum = torch.cumsum(diffs, dim=-1)                    # [B, T-1]
        counts = torch.arange(1, T, device=time_seqs.device,
                               dtype=time_seqs.dtype).unsqueeze(0)  # [1, T-1]
        prefix_mean = cumsum / counts                           # [B, T-1]

        # divisor[0] = 1.0 (t_shifted[:,0] is always 0); divisor[i] = prefix_mean[i-1]
        ones = torch.ones(B, 1, device=time_seqs.device, dtype=time_seqs.dtype)
        divisor = torch.cat([ones, prefix_mean], dim=-1).clamp(min=1e-6)  # [B, T]

        return t_shifted / divisor'''

    NEW = '''    def _normalize_timestamps(self, time_seqs):
        """Pass through timestamps already normalized by to_tensors (mean gap = 1.0).

        The upstream pipeline (to_tensors) already divides all inter-event gaps
        by their sequence-level mean, so time_seqs arrives with mean gap ~1.0.
        The previous prefix-mean divisor algebraically cancelled to [0,1,2,...,T-1],
        destroying temporal information. This version preserves the structure.
        """
        return time_seqs - time_seqs[:, :1]'''

    if OLD in code:
        code = code.replace(OLD, NEW)
        with open(hothp_path, 'w') as f:
            f.write(code)
        print('✓ Correção aplicada em torch_hothp.py')
    elif 'return time_seqs - time_seqs[:, :1]' in code:
        print('✓ Correção já estava aplicada')
    else:
        print('⚠ Padrão não encontrado — verifique torch_hothp.py manualmente')

    # Fix de importação do __init__
    init_path = 'ufc-easytpp/easy_tpp/model/__init__.py'
    with open(init_path, 'w') as f:
        f.write("""from easy_tpp.model.torch_model.torch_basemodel import TorchBaseModel
from easy_tpp.model.torch_model.torch_thp import THP as TorchTHP
from easy_tpp.model.torch_model.torch_rothp import RoTHP as TorchRoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP as TorchHoTHP
""")

    sys.path.insert(0, os.path.abspath('ufc-easytpp'))
    os.system('pip install omegaconf -q')
else:
    ROOT = os.path.abspath('..')
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)

print('✓ Setup concluído')

In [ ]:
# ── Imports do easy_tpp ───────────────────────────────────────────────────────
import easy_tpp.model.torch_model.torch_baselayer as baselayer

def _attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        if mask.dim() == 3:
            mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p = dropout(p)
    return torch.matmul(p, value), p

baselayer.attention = _attention

import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = _attention

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

COR_ROTHP = '#4C72B0'
COR_HOTHP = '#C44E52'

print(f'Device: {device}')
print('Imports OK')

In [ ]:
# ── Configurações ────────────────────────────────────────────────────────────
TRAIN_LEN      = 50        # eventos no treino
EXTRAP_FACTORS = [1, 2, 5, 10]
N_SEEDS        = 5
EPOCHS         = 300
PATIENCE       = 30
NUM_TYPES      = 2
PAD_ID         = NUM_TYPES

REGIMES = {
    'Decaimento lento  (β=0.025)': dict(
        mu    = np.array([0.4, 0.4]),
        alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
        beta  = 0.025,
    ),
    'Decaimento rápido (β=0.50)': dict(
        mu    = np.array([0.4, 0.4]),
        alpha = np.array([[0.12, 0.08], [0.08, 0.12]]),
        beta  = 0.50,
    ),
}

config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES, 'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': PAD_ID, 'time_emb_size': 32, 'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'Val',
    'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                 'patience_counter': 5, 'num_samples_boundary': 5,
                 'dtime_max': 5.0, 'num_step_gen': 1},
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})

print(f'TRAIN_LEN={TRAIN_LEN}, fatores={EXTRAP_FACTORS}, seeds={N_SEEDS}')

In [ ]:
# ── Funções auxiliares ───────────────────────────────────────────────────────

def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    for _ in range(200):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9: break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon: break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


def to_tensors(seqs):
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)
        t = (t - t[0]) / mg
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


def collate(batch, pad_id=PAD_ID):
    B = len(batch)
    L = max(len(x['time_seqs']) for x in batch)
    t_pad = torch.zeros(B, L)
    d_pad = torch.zeros(B, L)
    k_pad = torch.full((B, L), pad_id, dtype=torch.long)
    npm   = torch.zeros(B, L)
    causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
    attn   = torch.ones(B, L, L, dtype=torch.bool)
    for i, item in enumerate(batch):
        sl = len(item['time_seqs'])
        t_pad[i, :sl] = item['time_seqs']
        d_pad[i, :sl] = item['time_delta_seqs']
        k_pad[i, :sl] = item['type_seqs']
        npm[i, :sl] = 1.0
        m = causal.clone()
        m[:, sl:] = True; m[sl:, :] = True
        attn[i] = m
    return t_pad, d_pad, k_pad, npm, attn


def make_loader(data, bs, shuffle=False, seed=None):
    g = None
    if shuffle and seed is not None:
        g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(data, batch_size=min(bs, len(data)), shuffle=shuffle,
                      collate_fn=collate, generator=g)


def eval_nll(model, dl):
    model.eval()
    total_l = total_n = 0
    with torch.no_grad():
        for batch in dl:
            batch = [t.to(device) for t in batch]
            l, n = model.loglike_loss(batch)
            total_l += l.item(); total_n += n
    return total_l / (total_n + 1e-9)


def train_model(cls, train_dl, val_dl, lr, seed):
    set_seed(seed)
    m = cls(config).to(device)
    opt   = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=10, min_lr=1e-5)
    best_val, best_state, no_imp = float('inf'), None, 0
    for ep in range(EPOCHS):
        m.train()
        for batch in train_dl:
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            l, n = m.loglike_loss(batch)
            nll = l / (n + 1e-9)
            if not torch.isnan(nll):
                nll.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
        v = eval_nll(m, val_dl)
        sched.step(v)
        if v < best_val - 1e-4:
            best_val = v
            best_state = {k: v2.cpu().clone() for k, v2 in m.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break
    m.load_state_dict(best_state)
    return m, best_val

print('Funções prontas.')

In [ ]:
# ── Experimento principal ─────────────────────────────────────────────────────

all_results = {}   # regime → {factor → {'rothp': [...], 'hothp': [...]}}

for regime_name, proc in REGIMES.items():
    print(f'\n{"="*65}')
    print(f'Regime: {regime_name}')
    print(f'{"="*65}')

    mu, alpha, beta = proc['mu'], proc['alpha'], proc['beta']
    # horizon proporcional ao β para garantir sequências suficientes
    horizon_train = max(50.0, TRAIN_LEN / mu.sum() * 3)

    results = {f: {'rothp': [], 'hothp': []} for f in EXTRAP_FACTORS}

    for seed_idx in range(N_SEEDS):
        seed = 42 + seed_idx * 100
        rng  = np.random.default_rng(seed)

        # Dados de treino e validação
        raw_train = [simulate_hawkes(rng, mu, alpha, beta, horizon_train, 5, TRAIN_LEN)
                     for _ in range(400)]
        raw_val   = [simulate_hawkes(rng, mu, alpha, beta, horizon_train, 5, TRAIN_LEN)
                     for _ in range(100)]

        train_dl = make_loader(to_tensors(raw_train), 64, shuffle=True, seed=seed)
        val_dl   = make_loader(to_tensors(raw_val),   64)

        # Treina ambos os modelos
        rothp, rv = train_model(RoTHP, train_dl, val_dl, lr=1e-3, seed=seed)
        hothp, hv = train_model(HoTHP, train_dl, val_dl, lr=5e-4, seed=seed + 1)
        print(f'  Seed {seed_idx+1}: val RoTHP={rv:.4f}  HoTHP={hv:.4f}')

        # Avalia em cada fator de extrapolação
        for f in EXTRAP_FACTORS:
            target_len = TRAIN_LEN * f
            horizon_test = max(horizon_train * f, horizon_train + 10)
            raw_test = [simulate_hawkes(rng, mu, alpha, beta, horizon_test,
                                        TRAIN_LEN + 1, target_len)
                        for _ in range(200)]
            test_dl = make_loader(to_tensors(raw_test), 32)

            r_nll = eval_nll(rothp, test_dl)
            h_nll = eval_nll(hothp, test_dl)
            results[f]['rothp'].append(r_nll)
            results[f]['hothp'].append(h_nll)
            print(f'    α={f:>2}x  RoTHP={r_nll:.4f}  HoTHP={h_nll:.4f}  '
                  f'Δ={r_nll - h_nll:+.4f}')

    all_results[regime_name] = results

print('\nExperimento concluído.')

In [ ]:
# ── Tabela resumo ─────────────────────────────────────────────────────────────

print('\nRESUMO — NLL médio ± desvio (n=5 seeds)')
print('Δ = RoTHP - HoTHP  (positivo = HoTHP melhor)')

for regime_name, results in all_results.items():
    print(f'\n{regime_name}')
    print(f'  {"α":>4}  {"RoTHP":>14}  {"HoTHP":>14}  {"Δ":>10}')
    print(f'  {"-"*4}  {"-"*14}  {"-"*14}  {"-"*10}')
    for f in EXTRAP_FACTORS:
        r = np.array(results[f]['rothp'])
        h = np.array(results[f]['hothp'])
        delta = r - h
        print(f'  {f:>3}x  {r.mean():.4f}±{r.std():.4f}  '
              f'{h.mean():.4f}±{h.std():.4f}  {delta.mean():+.4f}')

In [ ]:
# ── Gráfico de linhas ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(EXTRAP_FACTORS))
xlabels = [f'{f}x' for f in EXTRAP_FACTORS]

for ax, (regime_name, results) in zip(axes, all_results.items()):
    for col, label, color in [('rothp', 'RoTHP', COR_ROTHP),
                               ('hothp', 'HoTHP (corrigido)', COR_HOTHP)]:
        means = np.array([np.mean(results[f][col]) for f in EXTRAP_FACTORS])
        stds  = np.array([np.std(results[f][col])  for f in EXTRAP_FACTORS])
        ax.plot(x, means, 'o-', color=color, lw=2, ms=7, label=label)
        ax.fill_between(x, means - stds, means + stds, color=color, alpha=0.15)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels)
    ax.set_xlabel('Fator de extrapolação (α)', fontsize=12)
    ax.set_ylabel('NLL (nats)', fontsize=12)
    ax.set_title(regime_name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(f'Train Short, Test Long — TRAIN_LEN={TRAIN_LEN}, n={N_SEEDS} seeds\n'
             f'HoTHP com _normalize_timestamps corrigido',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('validacao_pos_correcao.png', dpi=150, bbox_inches='tight')
plt.show()